# Enron Email Dataset — Topic Modelling Project (Email 5)

This notebook is an improved version of the previous pipeline.  
It **keeps the same project logic and structure**, but strengthens a few weak points:

- more careful normalization of noisy email artifacts,
- safer and better documented duplicate handling,
- slightly stronger stopword strategy for residual conversational noise,
- an explicit model-selection block with **perplexity + coherence + topic diversity**,
- a cleaner final interpretation section.

The goal is still the same: **prepare the Enron email corpus and identify the main themes of communication using topic modelling**.


In [6]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import kagglehub

from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

pd.set_option("display.max_colwidth", 200)
RANDOM_STATE = 42


/Users/vasya/keras_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the dataset

The original Enron corpus contains raw email strings, including technical headers, forwarded chains, signatures, and other noise.  
The first task is to load the data and confirm the basic structure.


In [7]:
dataset_path = kagglehub.dataset_download("wcukierski/enron-email-dataset")
csv_path = os.path.join(dataset_path, "emails.csv")

df = pd.read_csv(csv_path)

print("Dataset folder:", dataset_path)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing messages:", df["message"].isna().sum())
print("Duplicate rows:", df.duplicated().sum())

df.head()


Dataset folder: /Users/vasya/.cache/kagglehub/datasets/wcukierski/enron-email-dataset/versions/2
Shape: (517401, 2)
Columns: ['file', 'message']
Missing messages: 0
Duplicate rows: 0


,file,message
0,allen-p/_sent_mail/1.,"Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nConte..."
1,allen-p/_sent_mail/10.,"Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May 2001 13:51:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: john.lavorato@enron.com\nSubject: Re:\nMime-Version: 1.0\n..."
2,allen-p/_sent_mail/100.,"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme>\nDate: Wed, 18 Oct 2000 03:00:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: leah.arsdall@enron.com\nSubject: Re: test\nMime-Version: ..."
3,allen-p/_sent_mail/1000.,"Message-ID: <13505866.1075863688222.JavaMail.evans@thyme>\nDate: Mon, 23 Oct 2000 06:13:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: randall.gay@enron.com\nSubject: \nMime-Version: 1.0\nCont..."
4,allen-p/_sent_mail/1001.,"Message-ID: <30922949.1075863688243.JavaMail.evans@thyme>\nDate: Thu, 31 Aug 2000 05:07:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: greg.piper@enron.com\nSubject: Re: Hello\nMime-Version: 1..."


## 2. Text-cleaning functions

The raw Enron emails are not ready for topic modelling.  
I apply the following sequence:

- remove technical headers  
- keep only the main message body  
- remove signatures, disclaimers, and contact noise  
- normalize text for modelling  
- flag probable newsletters / administrative emails


In [8]:
def remove_top_email_header(text):
    """Remove the technical top header from the raw Enron email."""
    if pd.isna(text):
        return ""

    text = str(text).replace("\r\n", "\n").replace("\r", "\n").strip()
    if not text:
        return ""

    m = re.search(r'(?im)^X-FileName:.*$\n?', text)
    if m:
        return text[m.end():].strip()

    lines = text.split("\n")
    header_prefixes = (
        "Message-ID:", "Date:", "From:", "To:", "Subject:", "Mime-Version:",
        "Content-Type:", "Content-Transfer-Encoding:", "X-From:", "X-To:",
        "X-cc:", "X-bcc:", "X-Folder:", "X-Origin:", "X-FileName:"
    )

    body_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped == "":
            body_start = i + 1
            break

        if stripped.startswith(header_prefixes):
            continue

        body_start = i
        break

    return "\n".join(lines[body_start:]).strip()


def keep_only_main_message(text):
    """Keep the main content of the email and remove reply-chain wrappers."""
    if pd.isna(text):
        return ""

    text = str(text).replace("\r\n", "\n").replace("\r", "\n").strip()
    if not text:
        return ""

    is_forwarded = bool(re.search(r'(?im)forwarded by .* on \d{1,2}/\d{1,2}/\d{2,4}', text))

    if is_forwarded:
        lines = text.split("\n")
        cleaned_lines = []
        skip_metadata = True
        started_body = False

        for line in lines:
            stripped = line.strip()

            if re.search(r'(?i)forwarded by .* on \d{1,2}/\d{1,2}/\d{2,4}', stripped):
                continue
            if re.match(r'^\d{1,2}:\d{2}\s*(AM|PM)\s*-*$', stripped, flags=re.I):
                continue

            if skip_metadata and (
                re.match(r'(?i)^from:\s+', stripped) or
                re.match(r'(?i)^to:\s+', stripped) or
                re.match(r'(?i)^cc:\s*', stripped) or
                re.match(r'(?i)^bcc:\s*', stripped) or
                re.match(r'(?i)^subject:\s+', stripped) or
                re.match(r'(?i)^sent:\s+', stripped) or
                re.match(r'(?i)^date:\s+', stripped) or
                re.match(r'(?i)^".*?"\s*<[^>]+>.*$', stripped) or
                re.match(r'^[A-Z][a-z]+\s+[A-Z][a-z]+.*\d{1,2}/\d{1,2}/\d{2,4}', stripped) or
                re.match(r'^\d{1,2}/\d{1,2}/\d{2,4}.*$', stripped)
            ):
                continue

            if skip_metadata and stripped == "":
                continue

            if skip_metadata and stripped != "":
                skip_metadata = False
                started_body = True

            if started_body:
                cleaned_lines.append(line)

        text = "\n".join(cleaned_lines).strip()
        text = re.sub(r'(?m)^\s*>\s?', '', text)

    else:
        split_patterns = [
            r'(?im)^-+\s*original message\s*-+\s*$',
            r'(?im)^begin forwarded message:?\s*$',
            r'(?im)^on\s+.+wrote:\s*$',
            r'(?im)^-+\s*forwarded by\s+.*$'
        ]

        cut_positions = []
        for pattern in split_patterns:
            m = re.search(pattern, text)
            if m:
                cut_positions.append(m.start())

        if cut_positions:
            text = text[:min(cut_positions)].strip()

        text = re.sub(r'(?m)^\s*>\s?.*$', '', text)

    text = re.sub(r'(?im)^(to:|cc:|bcc:|subject:|from:|sent:|date:).*$\n?', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


def remove_signature_and_noise(text):
    """Remove signatures, disclaimers, contacts, URLs, HTML remnants, and footer noise."""
    if pd.isna(text):
        return ""

    text = str(text).strip()
    if not text:
        return ""

    text = re.sub(r'(?im)<embedded[^>]*>', ' ', text)
    text = re.sub(r'(?im)<[^>]+>', ' ', text)

    disclaimer_patterns = [
        r'(?im)^this message may contain confidential.*$',
        r'(?im)^the information contained in this e-mail.*$',
        r'(?im)^internet communications are not secure.*$',
        r'(?im)^please consider the environment before printing.*$',
        r'(?im)^this e-mail and any attachments are confidential.*$',
        r'(?im)^if you are not the intended recipient.*$'
    ]

    cut_positions = []
    for pattern in disclaimer_patterns:
        m = re.search(pattern, text)
        if m:
            cut_positions.append(m.start())

    if cut_positions:
        text = text[:min(cut_positions)].strip()

    text = re.sub(r'(?im)^(phone|fax|cell|mobile|pager|tel|telephone):.*$', ' ', text)
    text = re.sub(r'(?im)^.*\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b.*$', ' ', text)
    text = re.sub(r'(?im)^.*@\S+.*$', ' ', text)
    text = re.sub(r'(?im)^http\S+.*$', ' ', text)
    text = re.sub(r'(?im)^www\.\S+.*$', ' ', text)
    text = re.sub(r'(?m)^[-_=]{2,}\s*$', ' ', text)
    text = re.sub(r'(?im)^(attachment|attachments):.*$', ' ', text)
    text = re.sub(r'(?im)^inline attachment follows.*$', ' ', text)

    signature_markers = [
        r'(?im)^thanks[,]?\s*$',
        r'(?im)^thank you[,]?\s*$',
        r'(?im)^regards[,]?\s*$',
        r'(?im)^best[,]?\s*$',
        r'(?im)^best regards[,]?\s*$',
        r'(?im)^sincerely[,]?\s*$',
        r'(?im)^cheers[,]?\s*$'
    ]

    sig_positions = []
    for pattern in signature_markers:
        m = re.search(pattern, text)
        if m:
            sig_positions.append(m.start())

    if sig_positions:
        text = text[:min(sig_positions)].strip()

    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)

    return text.strip()


def normalize_for_topic_model(text):
    """Normalize text for topic modelling and suppress residual technical artifacts."""
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)

    # remove common email / HTML / encoding remnants
    text = re.sub(r'\b(?:font|html|width|height|align|border|cellpadding|cellspacing|gif|jpeg|png|nbsp)\b', ' ', text)
    text = re.sub(r'\b(?:forwarded|original message|inline attachment follows|attachment|attachments)\b', ' ', text)
    text = re.sub(r'\b(?:pm|am|mon|tue|wed|thu|thur|fri|sat|sun)\b', ' ', text)

    # keep alphabetic tokens only: years / IDs / codes usually add noise here
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\b[a-z]\b', ' ', text)
    text = re.sub(r'\b(?:image|images|id|type|final|vince|database|program)\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def is_probably_newsletter(text):
    if pd.isna(text):
        return False

    t = str(text).lower()
    indicators = 0

    if len(t.split()) > 1000:
        indicators += 1
    if t.count("http") > 3:
        indicators += 1
    if "unsubscribe" in t:
        indicators += 1
    if "click here" in t:
        indicators += 1
    if "newsletter" in t:
        indicators += 1
    if "view this email" in t:
        indicators += 1

    return indicators >= 2


def is_admin_or_survey(text):
    if pd.isna(text):
        return False

    t = str(text).lower()
    keywords = [
        "full name",
        "login id",
        "office location",
        "distribution groups",
        "shared calendar",
        "email calendar",
        "what type of computer do you have",
        "do you have a pda",
        "normal work hours",
        "survey",
        "questionnaire",
        "please complete"
    ]

    return sum(kw in t for kw in keywords) >= 2


## 3. Quick audit of the cleaning pipeline

I inspect a small sample of emails to verify that the cleaning functions remove headers and obvious noise without destroying the core message.


In [9]:
sample_idx = [1, 9, 15, 20, 50, 100, 200]

for idx in sample_idx:
    print("=" * 100)
    print(f"INDEX: {idx}")
    print("=" * 100)

    original = df.loc[idx, "message"]
    body = remove_top_email_header(original)
    main = keep_only_main_message(body)
    clean = remove_signature_and_noise(main)

    print("ORIGINAL:")
    print(original[:1200])

    print("\n" + "-" * 100)
    print("CLEANED:")
    print(clean[:1200])
    print("\n")


INDEX: 1
ORIGINAL:
Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>
Date: Fri, 4 May 2001 13:51:00 -0700 (PDT)
From: phillip.allen@enron.com
To: john.lavorato@enron.com
Subject: Re:
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: John J Lavorato <John J Lavorato/ENRON@enronXgate@ENRON>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Traveling to have a business meeting takes the fun out of the trip.  Especially if you have to prepare a presentation.  I would suggest holding the business plan meetings here then take a trip without any formal business meetings.  I would even try and get some honest opinions on whether a trip is even desired or necessary.

As far as the business meetings, I think it would be more productive to try and stimulate discussions across the different groups about what is working and what is 

## 4. Build the cleaned corpus

This stage keeps the original logic, but makes the diagnostic output a bit stronger.
We build several intermediate text versions and keep them for audit:

- `body_text` – after technical header removal,
- `main_message` – after reply-chain trimming,
- `clean_message` – after signatures / footers / obvious noise removal,
- `model_text` – normalized text used for topic modelling.

The goal is to keep the pipeline transparent and debuggable.


In [10]:
df["body_text"] = df["message"].apply(remove_top_email_header)
df["main_message"] = df["body_text"].apply(keep_only_main_message)
df["clean_message"] = df["main_message"].apply(remove_signature_and_noise)
df["model_text"] = df["clean_message"].apply(normalize_for_topic_model)

df["body_word_len"] = df["body_text"].str.split().str.len()
df["main_word_len"] = df["main_message"].str.split().str.len()
df["clean_word_len"] = df["clean_message"].str.split().str.len()
df["model_word_len"] = df["model_text"].str.split().str.len()

df["newsletter_flag"] = df["clean_message"].apply(is_probably_newsletter)
df["admin_flag"] = df["clean_message"].apply(is_admin_or_survey)

print("FINAL DATASET READY")
print("Rows in original df:", len(df))
print()
print(df["model_word_len"].describe())
print()
print("Newsletter-like rows:", int(df["newsletter_flag"].sum()))
print("Admin / survey rows:", int(df["admin_flag"].sum()))


FINAL DATASET READY
Rows in original df: 517401

count    517401.000000
mean        182.193944
std         788.127168
min           0.000000
25%          23.000000
50%          63.000000
75%         156.000000
max      136311.000000
Name: model_word_len, dtype: float64

Newsletter-like rows: 11793
Admin / survey rows: 474


### Filtering decision

The original project removed short texts, newsletters, and admin / survey content.  
This improved version keeps the same logic, but makes duplicate handling more explicit:

- filtering is done **before modelling**,
- duplicate removal is still **exact-text deduplication** on `model_text`,
- we also report how large the duplicate block is, because this matters for methodological transparency.


In [11]:
df_model = df[
    (df["model_word_len"] >= 10) &
    (df["model_word_len"] <= 1000) &
    (~df["newsletter_flag"]) &
    (~df["admin_flag"])
].copy()

before_n = len(df_model)

duplicate_counts = (
    df_model["model_text"]
    .value_counts(dropna=False)
    .rename_axis("model_text")
    .reset_index(name="count")
)

repeated_templates = duplicate_counts[duplicate_counts["count"] > 1].sort_values("count", ascending=False)

df_model = df_model.drop_duplicates(subset=["model_text"]).copy()
after_n = len(df_model)

print("Before duplicate removal:", before_n)
print("After duplicate removal:", after_n)
print("Removed exact duplicates:", before_n - after_n)
print("Duplicate share (%):", round((before_n - after_n) / before_n * 100, 2))

print("\nTop repeated cleaned texts (for audit):")
repeated_templates.head(10)


Before duplicate removal: 441190
After duplicate removal: 191200
Removed exact duplicates: 249990
Duplicate share (%): 56.66

Top repeated cleaned texts (for audit):


,model_text,count
0,start date hourahead hour no ancillary schedules awarded no variances detected log messages,1700
1,start date hourahead hour no ancillary schedules awarded no variances detected log messages parsing file portland westdesk california scheduling iso,1374
2,start date hourahead hour hourahead schedule download failed manual intervention required,894
3,the report named east totals published as of is now available for viewing on the website,264
4,start date hourahead hour no ancillary schedules awarded no variances detected log messages error retrieving hourahead price data process continuing,257
5,start date hourahead hour hourahead schedule download failed manual intervention required log messages error dbcaps data cannot perform this operation on closed unknown alias dbcaps data unknown a...,249
6,this warning is sent automatically to inform you that your mailbox is approaching the maximum size limit your mailbox size is currently kb mailbox size limits when your mailbox reaches kb you will...,232
7,following please find the daily enrononline executive summary enrononline executive summary for transaction summary external transactions today average daily external transactions day trailing avg...,230
8,task assignment status completed task priority task due on task start date,194
9,get your free download of msn explorer at report xls,161


## 5. Prepare a stronger modelling subset

We keep the same modelling idea, but make the subset slightly stricter:

- only reasonably substantial emails are used,
- extremely short emails are excluded because they often contribute generic conversational noise,
- the corpus is reset cleanly before vectorization.


In [12]:
df_topics_input = df_model[
    (df_model["model_word_len"] >= 40) &
    (df_model["model_word_len"] <= 800)
].copy()

docs = df_topics_input["model_text"].dropna().astype(str)
docs = docs[docs.str.strip() != ""].reset_index(drop=True)

print("Number of documents used for modelling:", len(docs))
docs.head()


Number of documents used for modelling: 130058


0    traveling to have business meeting takes the fun out of the trip especially if you have to prepare presentation would suggest holding the business plan meetings here then take trip without any for...
1    phillip as discussed during our phone conversation in parallon microturbine power generation deal for national accounts customer developing proposal to sell power to customer at fixed or collar fl...
2    lucy here are the rentrolls open them and save in the rentroll folder follow these steps so you don misplace these files click on save as click on the drop down triangle under save in click on the...
3    please respond to westgate enclosed are demographics on the westgate site from investor alliance investor alliance says that these demographics are similar to the package on san marcos that you re...
4    liane as we discussed yesterday concerned there may have been an attempt to manipulate the el paso san juan monthly index it appears that single buyer entered the marketplace 

In [13]:
from collections import Counter
import pandas as pd

def frequent_terms_by_length(series, min_len=1, max_len=3, top_n=50):
    tokens = " ".join(series.dropna()).split()
    counts = Counter(
        token for token in tokens
        if min_len <= len(token) <= max_len
    )
    return (
        pd.DataFrame(counts.items(), columns=["term", "count"])
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
        .head(top_n)
    )

print("Most frequent terms with length 1–3:")
display(frequent_terms_by_length(df["clean_message"], min_len=1, max_len=3, top_n=50))

print("Most frequent terms with length 4:")
display(frequent_terms_by_length(df["clean_message"], min_len=4, max_len=4, top_n=50))

print("Most frequent terms with length 5:")
display(frequent_terms_by_length(df["clean_message"], min_len=5, max_len=5, top_n=50))

Most frequent terms with length 1–3:


,term,count
0,the,4124837
1,to,2845947
2,and,2012339
3,of,1906952
4,a,1463990
5,in,1310789
6,for,1097876
7,is,947253
8,on,827577
9,you,746620


Most frequent terms with length 4:


,term,count
0,that,846929
1,will,601764
2,with,597732
3,this,549572
4,have,549152
5,from,406968
6,your,351218
7,been,158782
8,said,155927
9,they,151589


Most frequent terms with length 5:


,term,count
0,Enron,321199
1,would,261154
2,about,167946
3,which,167176
4,power,158455
5,their,148015
6,other,122870
7,these,100962
8,could,95626
9,there,92701


## 6. Shared helpers for modelling

This is still the same modelling block, but with two improvements:

1. a slightly stronger stopword set for residual conversational and technical noise;
2. an additional **topic diversity** metric, so model choice is not based on perplexity alone.


In [14]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stopwords = set(ENGLISH_STOP_WORDS).union({
    # company / email metadata noise
    "enron", "ect", "hou", "corp", "com", "mail", "email", "message",
    "original", "attached", "attachment", "subject", "cc", "bcc",
    "fwd", "fw", "re", "forwarded",

    # common email politeness / conversational noise
    "thanks", "thank", "please", "let", "know", "need",
    "pm", "am", "ll", "okay", "ok", "yes", "no",

    # relative-time conversational noise
    "today", "tomorrow", "yesterday",

    # generic conversational verbs
    "said", "going", "want", "make", "made", "getting",
    "got", "sent", "send",

    # frequent person-name noise
    "phillip", "allen", "john", "randy", "david", "dave", "mike",
    "ina", "julie", "paula", "brenda", "cynthia",

    # weak topic words that were repeatedly noisy
    "question", "questions"
})

extra_stopwords = {
    # weak conversational / stylistic noise
    "did", "really", "right", "way", "little", "come", "look",
    "home", "night", "sure",

    # generic document / admin noise
    "following", "comments", "file", "doc", "copy", "draft", "list",

    # short-token / contraction artifacts
    "don", "ve", "th", "eb"
}

disclaimer_stopwords = {
    # disclaimer / html / web artifacts
    "click", "intended", "recipient", "web", "site",
    "link", "links", "error", "receive",
    "font", "size", "iso", "trans", "td", "http",
    "www", "html", "width", "height", "align", "border",
    "cellpadding", "cellspacing", "color", "gif", "images",
    "image", "id", "type"
}

final_stopwords = sorted(
    custom_stopwords
    .union(extra_stopwords)
    .union(disclaimer_stopwords)
)


def build_count_vectorizer():
    return CountVectorizer(
        stop_words=final_stopwords,
        max_df=0.35,
        min_df=20,
        ngram_range=(1, 2),
        max_features=5000,
        token_pattern=r'(?u)\b[a-z][a-z]+\b'
    )


def build_tfidf_vectorizer():
    return TfidfVectorizer(
        stop_words=final_stopwords,
        max_df=0.35,
        min_df=20,
        ngram_range=(1, 2),
        max_features=5000,
        token_pattern=r'(?u)\b[a-z][a-z]+\b'
    )


def get_topic_words(model, feature_names, n_top_words=15):
    topic_word_lists = []
    for topic in model.components_:
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        topic_word_lists.append(top_words)
    return topic_word_lists


def print_topics(model, feature_names, n_top_words=15):
    topic_word_lists = get_topic_words(model, feature_names, n_top_words=n_top_words)
    for topic_idx, words in enumerate(topic_word_lists):
        print(f"TOPIC {topic_idx}:")
        print(", ".join(words))
        print("-" * 120)


def compute_topic_diversity(model, feature_names, top_n=15):
    """Share of unique words among all top topic words."""
    topic_word_lists = get_topic_words(model, feature_names, n_top_words=top_n)
    flat_words = [word for topic in topic_word_lists for word in topic]
    return len(set(flat_words)) / len(flat_words)


def summarize_top_terms(doc_series, n=30):
    counts = Counter(" ".join(doc_series).split())
    return pd.DataFrame(counts.most_common(n), columns=["term", "count"])


## 7. Baseline LDA model

We keep the simple baseline LDA, but now it uses the strengthened vectorizer settings from the helper block.


In [15]:
baseline_vectorizer = build_count_vectorizer()
baseline_dtm = baseline_vectorizer.fit_transform(docs)
baseline_feature_names = baseline_vectorizer.get_feature_names_out()

lda_baseline = LatentDirichletAllocation(
    n_components=3,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=30
)

lda_baseline.fit(baseline_dtm)

print("Baseline document-term matrix shape:", baseline_dtm.shape)
print_topics(lda_baseline, baseline_feature_names, n_top_words=20)


Baseline document-term matrix shape: (130058, 5000)
TOPIC 0:
energy, california, company, state, ferc, power, information, confidential, employees, commission, utility, million, consumers, use, electricity, utilities, bankruptcy, received, new, stock
------------------------------------------------------------------------------------------------------------------------
TOPIC 1:
time, like, just, meeting, houston, new, week, good, think, work, day, free, information, group, people, office, great, friday, hope, year
------------------------------------------------------------------------------------------------------------------------
TOPIC 2:
gas, agreement, market, power, deal, new, price, information, trading, energy, credit, time, contract, ena, day, review, report, business, changes, company
------------------------------------------------------------------------------------------------------------------------


## 8. Compare candidate LDA models

Instead of relying only on perplexity, this block compares candidate models on:

- **perplexity**,
- **topic diversity** (less overlap between topics is better),
- qualitative interpretability from the printed topic words.


In [16]:
vectorizer_compare = build_count_vectorizer()
dtm_compare = vectorizer_compare.fit_transform(docs)
feature_names_compare = vectorizer_compare.get_feature_names_out()

results = []
lda_models = {}
topic_options = [3, 5, 7, 10]

for n_topics in topic_options:
    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=RANDOM_STATE,
        learning_method="batch",
        max_iter=40
    )

    lda_model.fit(dtm_compare)
    lda_models[n_topics] = lda_model

    perplexity_value = lda_model.perplexity(dtm_compare)
    topic_diversity_value = compute_topic_diversity(lda_model, feature_names_compare, top_n=15)

    results.append({
        "n_topics": n_topics,
        "perplexity": perplexity_value,
        "topic_diversity": round(topic_diversity_value, 4)
    })

    print("=" * 120)
    print(f"LDA WITH {n_topics} TOPICS")
    print("=" * 120)
    print(f"Perplexity: {perplexity_value:.2f}")
    print(f"Topic diversity: {topic_diversity_value:.4f}\n")
    print_topics(lda_model, feature_names_compare, n_top_words=15)
    print("\n")

results_df = pd.DataFrame(results).sort_values(["perplexity", "topic_diversity"], ascending=[True, False]).reset_index(drop=True)
results_df


LDA WITH 3 TOPICS
Perplexity: 2205.42
Topic diversity: 0.8667

TOPIC 0:
energy, california, company, state, power, confidential, information, ferc, employees, commission, utility, million, consumers, electricity, use
------------------------------------------------------------------------------------------------------------------------
TOPIC 1:
time, like, meeting, just, houston, new, week, good, think, work, day, information, free, group, people
------------------------------------------------------------------------------------------------------------------------
TOPIC 2:
gas, agreement, market, power, deal, new, price, information, trading, credit, contract, time, energy, ena, review
------------------------------------------------------------------------------------------------------------------------


LDA WITH 5 TOPICS
Perplexity: 1964.48
Topic diversity: 0.8133

TOPIC 0:
employees, confidential, company, information, energy, california, folder, privileged, sender, consumers, fut

,n_topics,perplexity,topic_diversity
0,10,1722.038862,0.8067
1,7,1874.872213,0.8190
2,5,1964.483060,0.8133
3,3,2205.415957,0.8667


## 9. NMF as a comparison model

NMF is kept as an alternative topic model so the analysis is not dependent on a single method.


In [17]:
tfidf_vectorizer = build_tfidf_vectorizer()
tfidf = tfidf_vectorizer.fit_transform(docs)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

nmf_model = NMF(
    n_components=7,
    random_state=RANDOM_STATE,
    init="nndsvda",
    max_iter=400
)

W = nmf_model.fit_transform(tfidf)

print("=" * 120)
print("NMF WITH 7 TOPICS")
print("=" * 120)
print_topics(nmf_model, tfidf_feature_names, n_top_words=15)

dominant_topics_nmf = np.argmax(W, axis=1)
topic_distribution_nmf = pd.Series(dominant_topics_nmf).value_counts().sort_index()

topic_distribution_nmf_df = pd.DataFrame({
    "topic": topic_distribution_nmf.index,
    "document_count": topic_distribution_nmf.values,
    "share_percent": (topic_distribution_nmf.values / len(docs) * 100).round(2)
})

topic_distribution_nmf_df


NMF WITH 7 TOPICS
TOPIC 0:
power, energy, market, gas, new, trading, california, business, company, ferc, information, services, risk, group, order
------------------------------------------------------------------------------------------------------------------------
TOPIC 1:
employees, consumers, company, california, company declared, declared bankruptcy, company employees, energy bills, donate, declared, retirement, millions, bills, energy, funds
------------------------------------------------------------------------------------------------------------------------
TOPIC 2:
agreement, sara, north america, america, north, smith, houston, smith street, isda, houston texas, street houston, street, texas, master, america smith
------------------------------------------------------------------------------------------------------------------------
TOPIC 3:
time, meeting, just, like, good, think, week, work, hope, day, great, friday, talk, morning, people
----------------------------------

,topic,document_count,share_percent
0,0,32341,24.87
1,1,1650,1.27
2,2,15202,11.69
3,3,52354,40.25
4,4,9005,6.92
5,5,3822,2.94
6,6,15684,12.06


## 10. Remove obvious residual noise

This is still a light post-cleaning step, not a full new pipeline.
The purpose is only to remove clearly non-substantive residual texts that survived the earlier rules.


In [18]:
noise_patterns = [
    "web specials",
    "thank you for subscribing",
    "inline attachment follows",
    "outlook migration",
    "calendar entry",
    "view this email",
    "unsubscribe",
    "distribution groups",
    "shared calendar",
    "please complete",
    "questionnaire"
]

mask_keep = ~docs.str.contains("|".join(noise_patterns), case=False, na=False, regex=True)
docs_denoised = docs[mask_keep].reset_index(drop=True)

print("Original number of documents:", len(docs))
print("Remaining after removing obvious noise:", len(docs_denoised))
print("Removed documents:", len(docs) - len(docs_denoised))
print("Share removed (%):", round((len(docs) - len(docs_denoised)) / len(docs) * 100, 2))
print()
print("Most frequent terms after denoising audit:")
summarize_top_terms(docs_denoised, n=25)


Original number of documents: 130058
Remaining after removing obvious noise: 127207
Removed documents: 2851
Share removed (%): 2.19

Most frequent terms after denoising audit:


,term,count
0,the,999316
1,to,691412
2,and,460917
3,of,391877
4,you,300734
5,in,292802
6,for,271675
7,is,243269
8,that,217240
9,on,203981


## 11. Final LDA model on the denoised corpus

The final model keeps the original project idea of using **7 topics**, but now on a slightly better cleaned corpus and with stronger modelling settings.

Why still 7 topics?
- it usually gives a better balance than 3 or 5,
- it is often more interpretable than 10 when corporate email topics start to fragment,
- it remains easy to explain in a final report.


In [19]:
vectorizer_final = build_count_vectorizer()
dtm_final = vectorizer_final.fit_transform(docs_denoised)
feature_names_final = vectorizer_final.get_feature_names_out()

lda_final = LatentDirichletAllocation(
    n_components=7,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=40
)

doc_topic_matrix_final = lda_final.fit_transform(dtm_final)

print("=" * 120)
print("FINAL LDA: 7 TOPICS ON DENOISED CORPUS")
print("=" * 120)
print(f"Perplexity: {lda_final.perplexity(dtm_final):.2f}")
print(f"Topic diversity: {compute_topic_diversity(lda_final, feature_names_final, top_n=15):.4f}\n")
print_topics(lda_final, feature_names_final, n_top_words=15)

dominant_topics_final = np.argmax(doc_topic_matrix_final, axis=1)
topic_distribution_final = pd.Series(dominant_topics_final).value_counts().sort_index()

topic_distribution_final_df = pd.DataFrame({
    "topic": topic_distribution_final.index,
    "document_count": topic_distribution_final.values,
    "share_percent": (topic_distribution_final.values / len(docs_denoised) * 100).round(2)
})

topic_distribution_final_df


FINAL LDA: 7 TOPICS ON DENOISED CORPUS
Perplexity: 1838.50
Topic diversity: 0.8381

TOPIC 0:
just, think, like, good, time, work, hope, day, great, week, people, things, new, talk, weekend
------------------------------------------------------------------------------------------------------------------------
TOPIC 1:
power, market, energy, business, new, risk, company, issues, ferc, group, management, information, trading, year, services
------------------------------------------------------------------------------------------------------------------------
TOPIC 2:
new, information, free, received, service, internet, available, area, address, online, order, access, futures, content, price
------------------------------------------------------------------------------------------------------------------------
TOPIC 3:
agreement, credit, legal, information, review, master, ena, letter, sara, america, north, north america, received, contract, isda
------------------------------------------

,topic,document_count,share_percent
0,0,21478,16.88
1,1,23762,18.68
2,2,8871,6.97
3,3,25997,20.44
4,4,1651,1.30
5,5,24523,19.28
6,6,20925,16.45


## 12. Representative emails for each final topic

Top words are useful, but they are not enough on their own.  
To interpret each topic more reliably, I inspect the highest-scoring representative emails.


In [20]:
topic_word_lists_final = get_topic_words(lda_final, feature_names_final, n_top_words=12)

n_examples = 3
for topic_idx in range(7):
    print("=" * 140)
    print(f"TOPIC {topic_idx}")
    print("=" * 140)
    print("TOP WORDS:", ", ".join(topic_word_lists_final[topic_idx]))
    print()

    top_doc_indices = np.argsort(doc_topic_matrix_final[:, topic_idx])[::-1][:n_examples]

    for rank, doc_idx in enumerate(top_doc_indices, start=1):
        topic_score = doc_topic_matrix_final[doc_idx, topic_idx]
        print(f"[Representative email #{rank}]  document_position={doc_idx}  topic_score={topic_score:.4f}")
        print(docs_denoised.iloc[doc_idx][:2500])
        print("\n" + "-" * 140 + "\n")


TOPIC 0
TOP WORDS: just, think, like, good, time, work, hope, day, great, week, people, things

[Representative email #1]  document_position=23833  topic_score=0.9978
start date hourahead hour no ancillary schedules awarded variances detected variances detected in energy import export schedule log messages error retrieving hourahead price data process continuing energy import export schedule hour bad data from iso trans sc ectstsw mkt trans date tie point pverde devers interchg ciso epmi engy nfrm hour bad data from iso trans sc ectstsw mkt trans date tie point fcornr psuedo interchg epmi ciso engy firm hour bad data from iso trans sc ectstsw mkt trans date tie point mead walc interchg epmi ciso engy firm hour bad data from iso trans sc ectrt mkt trans date tie point malin rndmtn interchg epmi ciso aaa engy wheel hour bad data from iso trans sc ectrt mkt trans date tie point fcornr psuedo interchg epmi ciso aaa engy wheel hour bad data from iso trans sc ectrt mkt trans date tie point p

## 13. Final model-selection diagnostics

This block makes the model choice more defensible by adding **coherence** on the denoised corpus.
Together with perplexity and topic diversity, this gives a more balanced basis for selecting the final solution.


In [21]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel


def compute_lda_diagnostics(doc_series, n_topics):
    vectorizer = build_count_vectorizer()
    dtm = vectorizer.fit_transform(doc_series)
    feature_names = vectorizer.get_feature_names_out()

    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=RANDOM_STATE,
        learning_method="batch",
        max_iter=40
    )
    lda_model.fit(dtm)

    topic_word_lists = get_topic_words(lda_model, feature_names, n_top_words=15)
    tokenized_docs = [doc.split() for doc in doc_series]
    dictionary = Dictionary(tokenized_docs)

    coherence_model = CoherenceModel(
        topics=topic_word_lists,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v"
    )

    return {
        "n_topics": n_topics,
        "perplexity": round(lda_model.perplexity(dtm), 2),
        "coherence_c_v": round(coherence_model.get_coherence(), 4),
        "topic_diversity": round(compute_topic_diversity(lda_model, feature_names, top_n=15), 4)
    }


diagnostics_df = pd.DataFrame(
    [compute_lda_diagnostics(docs_denoised, n) for n in [3, 5, 7, 10]]
).sort_values(
    ["coherence_c_v", "topic_diversity", "perplexity"],
    ascending=[False, False, True]
).reset_index(drop=True)

diagnostics_df


,n_topics,perplexity,coherence_c_v,topic_diversity
0,10,1680.67,0.5844,0.8000
1,7,1838.50,0.5359,0.8381
2,5,1988.02,0.4855,0.8533
3,3,2201.50,0.4048,0.8667


## 14. Final iterative denoising and model refinement

After the initial denoising and model comparison, several topics were still dominated by residual non-substantive emails, such as personal forwards, service notifications, travel confirmations, and other low-information communication.  
To improve content validity, an additional targeted cleanup stage was introduced.

This final refinement step included:

- removal of residual informal and personal email patterns,
- stronger near-duplicate filtering,
- heuristic filtering of low-information documents,
- a final comparison of candidate topic numbers.

This iterative approach was important because statistical fit alone was not sufficient.  
Representative-document inspection showed that some high-scoring models still captured noise rather than meaningful business themes.  
Therefore, the final topic model was selected based on both quantitative diagnostics and qualitative interpretability.


In [23]:


import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

# ------------------------------------------------------------
# 0. Choose the best available base corpus automatically
# ------------------------------------------------------------

if "docs_stage2" in globals():
    base_docs = docs_stage2.copy()
elif "docs_denoised" in globals():
    base_docs = docs_denoised.copy()
elif "docs" in globals():
    base_docs = docs.copy()
else:
    raise ValueError("No source corpus found. Expected one of: docs_stage2, docs_denoised, or docs.")

base_docs = pd.Series(base_docs).dropna().astype(str).reset_index(drop=True)

print("Base documents loaded:", len(base_docs))

# ------------------------------------------------------------
# 1. Targeted final cleanup
# ------------------------------------------------------------

stage3_patterns = [
    # Travel confirmations / itineraries
    r"thank you for booking your travel",
    r"flight summary",
    r"economy coach class",
    r"boarding pass",
    r"ticket customers can also use",
    r"travel itinerary and receipt",
    r"depart houston",
    r"arrive houston",

    # Airline / travel promos
    r"dear savers subscriber",
    r"book online at",
    r"earn miles",
    r"dividend miles",
    r"vacations is pleased to offer",

    # Membership / formal notice distributions
    r"membership services",
    r"notice of intention to transfer",
    r"notice of intention to lease",
    r"switch of lessor",
    r"switch of status",
    r"reinstatement of membership privileges",

    # Chain / inspirational / nonsense
    r"long time ago there was huge apple tree",
    r"this is story of everyone",
    r"he my best friend very bright guy",

    # Vessel / shipping status reports
    r"\bposrep\b",
    r"position at noon",
    r"dist from last noon",
    r"eta next port",
    r"average sea weather",

    # Repetitive pipeline / allocation dumps
    r"quantity cut cd",
    r"nominated working capacity allocation",
    r"\barng eff date\b",
    r"\breceipt delivery quantity\b",

    # residual click-confirm / service messages
    r"please click on the link below to indicate you have received this email",
    r"if you click on the above line and nothing happens",

    # earlier junk families worth keeping here too
    r"confidentiality notice",
    r"legally privileged",
    r"if you are not an intended recipient",
    r"notify the sender immediately",
    r"delete this mail from your computer",
    r"virus warning",

    r"writing to urge you to donate",
    r"lost their retirement savings",
    r"unable to pay their energy bills",

    r"scheduled system outages",
    r"facility operations",
    r"environments impacted",
    r"\bbackout\b",
    r"server reboots",
    r"data center",

    r"synchronizing mailbox",
    r"synchronizing hierarchy",
    r"synchronizing folder",
    r"offline folder",
    r"view form updated",

    r"while supplies last",
    r"regular sale",
    r"click below",
    r"to be removed from .* email list",
    r"\bspecials\b",

    r"schedule download failed",
    r"manual intervention required",
    r"unknown alias",
    r"cannot perform this operation on closed",

    r"\bwpd\b(?:\s+\bwpd\b){5,}",
    r"inline attachment follows",
]

stage3_union = "|".join(stage3_patterns)

mask_keep = ~base_docs.str.contains(stage3_union, case=False, na=False, regex=True)
docs_stage3 = base_docs[mask_keep].reset_index(drop=True)

print("After targeted pattern cleanup:", len(docs_stage3))

# ------------------------------------------------------------
# 2. Heuristic quality filtering
# ------------------------------------------------------------

def doc_quality_features(text):
    tokens = re.findall(r"\b[a-z]+\b", str(text).lower())
    n_tokens = len(tokens)

    if n_tokens == 0:
        return pd.Series({
            "n_tokens": 0,
            "unique_ratio": 0.0,
            "top10_share": 1.0,
            "digit_ratio": 0.0
        })

    counts = pd.Series(tokens).value_counts()
    unique_ratio = len(set(tokens)) / n_tokens
    top10_share = counts.head(10).sum() / n_tokens
    digit_ratio = sum(ch.isdigit() for ch in str(text)) / max(len(str(text)), 1)

    return pd.Series({
        "n_tokens": n_tokens,
        "unique_ratio": unique_ratio,
        "top10_share": top10_share,
        "digit_ratio": digit_ratio
    })

quality_df = docs_stage3.apply(doc_quality_features)

mask_quality = (
    (quality_df["n_tokens"] >= 40) &
    (quality_df["unique_ratio"] >= 0.18) &
    (quality_df["top10_share"] <= 0.55) &
    (quality_df["digit_ratio"] <= 0.20)
)

docs_stage3 = docs_stage3[mask_quality].reset_index(drop=True)

print("After heuristic quality filtering:", len(docs_stage3))

# ------------------------------------------------------------
# 3. Stronger near-duplicate filtering
# ------------------------------------------------------------

def stronger_signature(text, n_tokens=160):
    text = str(text).lower()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = re.findall(r"\b[a-z]+\b", text)
    return " ".join(tokens[:n_tokens])

sig = docs_stage3.apply(stronger_signature)
sig_counts = sig.value_counts()

drop_signatures = set(sig_counts[sig_counts >= 5].index)
docs_stage3 = docs_stage3[~sig.isin(drop_signatures)].reset_index(drop=True)

print("After stronger near-duplicate filtering:", len(docs_stage3))
print("Dropped repeated signatures:", len(drop_signatures))

# ------------------------------------------------------------
# 4. Stopwords: inherit previous list if available, otherwise fallback
# ------------------------------------------------------------

if "final_stopwords_stage2" in globals():
    base_stopwords = set(final_stopwords_stage2)
elif "final_stopwords" in globals():
    base_stopwords = set(final_stopwords)
else:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    base_stopwords = set(ENGLISH_STOP_WORDS)

extra_stopwords_final = {
    "time", "new", "information", "work", "like", "just", "day", "week",
    "people", "best", "try", "say", "better", "office", "meeting",
    "conference", "schedule", "date", "free", "online", "service",
    "access", "number", "address", "thanks", "please", "help",
    "houston", "jeff", "steve", "mark"
}

final_stopwords_stage3 = sorted(base_stopwords.union(extra_stopwords_final))

# ------------------------------------------------------------
# 5. Vectorizer
# ------------------------------------------------------------

vectorizer_final3 = CountVectorizer(
    stop_words=final_stopwords_stage3,
    max_df=0.25,
    min_df=25,
    ngram_range=(1, 2),
    max_features=7000,
    token_pattern=r"(?u)\b[a-z][a-z]+\b"
)

dtm_final3 = vectorizer_final3.fit_transform(docs_stage3)
feature_names_final3 = vectorizer_final3.get_feature_names_out()

print("DTM shape:", dtm_final3.shape)

# ------------------------------------------------------------
# 6. Random state
# ------------------------------------------------------------

RANDOM_STATE_FINAL = globals().get("RANDOM_STATE", 42)

# ------------------------------------------------------------
# 7. Helpers
# ------------------------------------------------------------

def get_topic_words(model, feature_names, n_top_words=15):
    topic_word_lists = []
    for topic in model.components_:
        top_indices = topic.argsort()[-n_top_words:][::-1]
        topic_word_lists.append([feature_names[i] for i in top_indices])
    return topic_word_lists

def compute_topic_diversity(model, feature_names, top_n=15):
    topic_words = get_topic_words(model, feature_names, n_top_words=top_n)
    flat_words = [w for topic in topic_words for w in topic]
    return len(set(flat_words)) / len(flat_words)

def compute_topic_overlap(model, feature_names, top_n=15):
    topic_words = get_topic_words(model, feature_names, n_top_words=top_n)
    overlaps = []
    for i in range(len(topic_words)):
        for j in range(i + 1, len(topic_words)):
            s1 = set(topic_words[i])
            s2 = set(topic_words[j])
            overlaps.append(len(s1.intersection(s2)) / top_n)
    return np.mean(overlaps) if overlaps else 0.0

def compute_coherence_c_v(topic_word_lists, doc_series):
    tokenized_docs = [doc.split() for doc in doc_series]
    dictionary = Dictionary(tokenized_docs)
    coherence_model = CoherenceModel(
        topics=topic_word_lists,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v"
    )
    return coherence_model.get_coherence()

# ------------------------------------------------------------
# 8. Compare models
# ------------------------------------------------------------

topic_options = [4, 5, 6, 7, 8]
results = []
models = {}
doc_topic_matrices = {}

for n_topics in topic_options:
    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=RANDOM_STATE_FINAL,
        learning_method="batch",
        max_iter=50
    )

    doc_topic_matrix = lda_model.fit_transform(dtm_final3)

    models[n_topics] = lda_model
    doc_topic_matrices[n_topics] = doc_topic_matrix

    topic_word_lists = get_topic_words(lda_model, feature_names_final3, n_top_words=15)

    perplexity_value = lda_model.perplexity(dtm_final3)
    coherence_value = compute_coherence_c_v(topic_word_lists, docs_stage3)
    diversity_value = compute_topic_diversity(lda_model, feature_names_final3, top_n=15)
    overlap_value = compute_topic_overlap(lda_model, feature_names_final3, top_n=15)

    dominant_topics = np.argmax(doc_topic_matrix, axis=1)
    topic_shares = pd.Series(dominant_topics).value_counts(normalize=True).sort_index()
    smallest_topic_share = topic_shares.min()

    results.append({
        "n_topics": n_topics,
        "perplexity": round(perplexity_value, 2),
        "coherence_c_v": round(coherence_value, 4),
        "topic_diversity": round(diversity_value, 4),
        "topic_overlap": round(overlap_value, 4),
        "smallest_topic_share": round(float(smallest_topic_share), 4),
        "vocab_size": dtm_final3.shape[1]
    })

results_df = pd.DataFrame(results)

results_df["rank_perplexity"] = results_df["perplexity"].rank(method="min", ascending=True)
results_df["rank_coherence"] = results_df["coherence_c_v"].rank(method="min", ascending=False)
results_df["rank_diversity"] = results_df["topic_diversity"].rank(method="min", ascending=False)
results_df["rank_overlap"] = results_df["topic_overlap"].rank(method="min", ascending=True)
results_df["rank_smallest_share"] = results_df["smallest_topic_share"].rank(method="min", ascending=False)

results_df["rank_score"] = (
    results_df["rank_perplexity"] +
    2.0 * results_df["rank_coherence"] +
    1.5 * results_df["rank_diversity"] +
    1.5 * results_df["rank_overlap"] +
    1.0 * results_df["rank_smallest_share"]
)

results_df = results_df.sort_values(
    ["rank_score", "coherence_c_v", "topic_diversity"],
    ascending=[True, False, False]
).reset_index(drop=True)

print("\n" + "=" * 120)
print("FINAL MODEL COMPARISON")
print("=" * 120)
display(results_df[[
    "n_topics", "perplexity", "coherence_c_v", "topic_diversity",
    "topic_overlap", "smallest_topic_share", "vocab_size", "rank_score"
]])

best_n_topics = int(results_df.loc[0, "n_topics"])
best_lda = models[best_n_topics]
best_doc_topic = doc_topic_matrices[best_n_topics]

print(f"\nBest final model: {best_n_topics} topics")

# ------------------------------------------------------------
# 9. Final topics
# ------------------------------------------------------------

final_topic_words = get_topic_words(best_lda, feature_names_final3, n_top_words=15)

print("\n" + "=" * 120)
print("FINAL TOPICS")
print("=" * 120)

for topic_idx, words in enumerate(final_topic_words):
    print(f"TOPIC {topic_idx}:")
    print(", ".join(words))
    print("-" * 120)

# ------------------------------------------------------------
# 10. Topic distribution
# ------------------------------------------------------------

dominant_topics = np.argmax(best_doc_topic, axis=1)
topic_distribution = pd.Series(dominant_topics).value_counts().sort_index()

topic_distribution_df = pd.DataFrame({
    "topic": topic_distribution.index,
    "document_count": topic_distribution.values,
    "share_percent": (topic_distribution.values / len(docs_stage3) * 100).round(2)
})

print("\nTOPIC DISTRIBUTION")
display(topic_distribution_df)

# ------------------------------------------------------------
# 11. Representative emails
# ------------------------------------------------------------

n_examples = 3

for topic_idx in range(best_n_topics):
    print("=" * 140)
    print(f"TOPIC {topic_idx}")
    print("=" * 140)
    print("TOP WORDS:", ", ".join(final_topic_words[topic_idx]))
    print()

    top_doc_indices = np.argsort(best_doc_topic[:, topic_idx])[::-1][:n_examples]

    for rank, doc_idx in enumerate(top_doc_indices, start=1):
        topic_score = best_doc_topic[doc_idx, topic_idx]
        print(f"[Representative email #{rank}] document_position={doc_idx} topic_score={topic_score:.4f}")
        print(docs_stage3.iloc[doc_idx][:2500])
        print("\n" + "-" * 140 + "\n")

Base documents loaded: 121769
After targeted pattern cleanup: 121038
After heuristic quality filtering: 120099
After stronger near-duplicate filtering: 120033
Dropped repeated signatures: 10
DTM shape: (120033, 7000)

FINAL MODEL COMPARISON


,n_topics,perplexity,coherence_c_v,topic_diversity,topic_overlap,smallest_topic_share,vocab_size,rank_score
0,6,2558.73,0.5625,0.8778,0.0533,0.1014,7000,17.0
1,5,2624.63,0.5394,0.9200,0.0400,0.1068,7000,18.5
2,4,2733.49,0.4927,0.9333,0.0444,0.1843,7000,20.5
3,7,2496.23,0.5492,0.8571,0.0540,0.0807,7000,22.0
4,8,2431.25,0.5475,0.8500,0.0571,0.0638,7000,27.0



Best final model: 6 topics

FINAL TOPICS
TOPIC 0:
power, energy, california, market, ferc, state, customers, commission, electric, gas, electricity, utilities, pg, order, utility
------------------------------------------------------------------------------------------------------------------------
TOPIC 1:
think, year, check, guys, days, soon, sorry, looking, tell, phone, house, saturday, didn, big, probably
------------------------------------------------------------------------------------------------------------------------
TOPIC 2:
agreement, legal, master, sara, letter, america, north, north america, ena, isda, credit, smith, form, contract, changes
------------------------------------------------------------------------------------------------------------------------
TOPIC 3:
business, group, management, team, report, process, services, employees, company, issues, year, support, risk, plan, discuss
--------------------------------------------------------------------------------

,topic,document_count,share_percent
0,0,12176,10.14
1,1,26862,22.38
2,2,22740,18.94
3,3,28111,23.42
4,4,16988,14.15
5,5,13156,10.96


TOPIC 0
TOP WORDS: power, energy, california, market, ferc, state, customers, commission, electric, gas, electricity, utilities, pg, order, utility

[Representative email #1] document_position=17965 topic_score=0.9977
content transfer encoding quoted printable gene godley marc hebert paul fox california mime version content text plain charset iso content disposition inline administration helps electricity consumers by proposing reliability standards and working to lower costs clinton gore administration takes action to help californians secretary of energy bill richardson today announced series of initiatives that the clinton gore administration is taking to help california reduce the strain on their electricity system and protect consumers most significantly richardson said the administration will likely send proposed rule making to the federal energy regulatory commission ferc to establish mandatory reliability standards for electricity doing administratively what congress failed to 

## 15. Final micro-cleanup of residual informal communication

Although the previous refinement stage substantially improved topic quality, one topic still captured informal and weakly informative email traffic.  
To reduce this residual noise, an additional micro-cleanup step was applied. This stage removed recurring informal patterns and re-estimated the topic model to check whether a cleaner and more interpretable structure could be obtained.

In [25]:
# ============================================================
# OPTIONAL FINAL MICRO-CLEANUP FOR RESIDUAL INFORMAL EMAILS
# ============================================================

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

docs_stage4 = docs_stage3.copy()

residual_informal_patterns = [
    r"little johnny",
    r"god blessings to you all",
    r"joyful holiday season",
    r"long time ago there was huge apple tree",
    r"this is story of everyone",
    r"please pass this message along",
    r"the more the merrier",
    r"weekend we will be hosting",
    r"how could life get any better",
    r"spouses dogs kids roommates",
    r"friends whose email addresses",
    r"hope everyone can make it",
    r"beer",
    r"bbq",
    r"pumpkins",
    r"saturday",
    r"sunday",
    r"holiday season",
]

union_informal = "|".join(residual_informal_patterns)

mask_keep_stage4 = ~docs_stage4.str.contains(
    union_informal,
    case=False,
    na=False,
    regex=True
)

docs_stage4 = docs_stage4[mask_keep_stage4].reset_index(drop=True)

print("Documents after residual informal-email cleanup:", len(docs_stage4))

# stronger stopwords for this final pass
final_stopwords_stage4 = sorted(set(final_stopwords_stage3).union({
    "think", "year", "check", "guys", "days", "soon", "sorry",
    "looking", "tell", "phone", "house", "saturday", "didn",
    "big", "probably"
}))

vectorizer_stage4 = CountVectorizer(
    stop_words=final_stopwords_stage4,
    max_df=0.25,
    min_df=25,
    ngram_range=(1, 2),
    max_features=7000,
    token_pattern=r"(?u)\b[a-z][a-z]+\b"
)

dtm_stage4 = vectorizer_stage4.fit_transform(docs_stage4)
feature_names_stage4 = vectorizer_stage4.get_feature_names_out()

RANDOM_STATE_STAGE4 = globals().get("RANDOM_STATE", 42)

def get_topic_words_stage4(model, feature_names, n_top_words=15):
    topic_word_lists = []
    for topic in model.components_:
        top_indices = topic.argsort()[-n_top_words:][::-1]
        topic_word_lists.append([feature_names[i] for i in top_indices])
    return topic_word_lists

def compute_topic_diversity_stage4(model, feature_names, top_n=15):
    topic_words = get_topic_words_stage4(model, feature_names, n_top_words=top_n)
    flat_words = [w for topic in topic_words for w in topic]
    return len(set(flat_words)) / len(flat_words)

def compute_topic_overlap_stage4(model, feature_names, top_n=15):
    topic_words = get_topic_words_stage4(model, feature_names, n_top_words=top_n)
    overlaps = []
    for i in range(len(topic_words)):
        for j in range(i + 1, len(topic_words)):
            overlaps.append(len(set(topic_words[i]).intersection(set(topic_words[j]))) / top_n)
    return np.mean(overlaps) if overlaps else 0.0

def compute_coherence_stage4(topic_word_lists, docs_series):
    tokenized_docs = [doc.split() for doc in docs_series]
    dictionary = Dictionary(tokenized_docs)
    cm = CoherenceModel(
        topics=topic_word_lists,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v"
    )
    return cm.get_coherence()

results_stage4 = []
models_stage4 = {}
doc_topics_stage4 = {}

for n_topics in [5, 6]:
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=RANDOM_STATE_STAGE4,
        learning_method="batch",
        max_iter=50
    )
    doc_topic = lda.fit_transform(dtm_stage4)

    models_stage4[n_topics] = lda
    doc_topics_stage4[n_topics] = doc_topic

    topic_words = get_topic_words_stage4(lda, feature_names_stage4, 15)
    perplexity = lda.perplexity(dtm_stage4)
    coherence = compute_coherence_stage4(topic_words, docs_stage4)
    diversity = compute_topic_diversity_stage4(lda, feature_names_stage4, 15)
    overlap = compute_topic_overlap_stage4(lda, feature_names_stage4, 15)

    dominant = np.argmax(doc_topic, axis=1)
    shares = pd.Series(dominant).value_counts(normalize=True).sort_index()
    smallest_share = shares.min()

    results_stage4.append({
        "n_topics": n_topics,
        "perplexity": round(perplexity, 2),
        "coherence_c_v": round(coherence, 4),
        "topic_diversity": round(diversity, 4),
        "topic_overlap": round(overlap, 4),
        "smallest_topic_share": round(float(smallest_share), 4)
    })

results_stage4_df = pd.DataFrame(results_stage4).sort_values(
    ["coherence_c_v", "topic_overlap", "topic_diversity"],
    ascending=[False, True, False]
).reset_index(drop=True)

print("\nFINAL MICRO-CLEANUP MODEL COMPARISON")
display(results_stage4_df)

best_n_stage4 = int(results_stage4_df.loc[0, "n_topics"])
best_lda_stage4 = models_stage4[best_n_stage4]
best_doc_topic_stage4 = doc_topics_stage4[best_n_stage4]
best_words_stage4 = get_topic_words_stage4(best_lda_stage4, feature_names_stage4, 15)

print(f"\nBest model after micro-cleanup: {best_n_stage4} topics")

print("\nFINAL TOPICS AFTER MICRO-CLEANUP")
for i, words in enumerate(best_words_stage4):
    print(f"TOPIC {i}:")
    print(", ".join(words))
    print("-" * 100)

Documents after residual informal-email cleanup: 115160

FINAL MICRO-CLEANUP MODEL COMPARISON


,n_topics,perplexity,coherence_c_v,topic_diversity,topic_overlap,smallest_topic_share
0,6,2599.95,0.5391,0.8889,0.0489,0.0761
1,5,2681.74,0.5063,0.8933,0.0533,0.0571



Best model after micro-cleanup: 6 topics

FINAL TOPICS AFTER MICRO-CLEANUP
TOPIC 0:
business, group, management, team, risk, issues, process, discuss, working, trading, services, plan, employees, support, project
----------------------------------------------------------------------------------------------------
TOPIC 1:
deals, data, price, product, gas, report, daily, eol, trading, desk, deal, book, password, month, products
----------------------------------------------------------------------------------------------------
TOPIC 2:
content, visit, order, dinner, life, told, left, family, place, hear, long, lunch, hey, happy, money
----------------------------------------------------------------------------------------------------
TOPIC 3:
energy, power, california, market, state, company, ferc, gas, electric, commission, electricity, natural, public, natural gas, utilities
----------------------------------------------------------------------------------------------------
TOPIC 4:
g

## 16. Final topic interpretation

The final model selected **6 topics** as the most interpretable structure after iterative denoising and targeted micro-cleanup.

The resulting topics can be interpreted as follows:

1. **Internal Management and Risk Coordination**  
   This topic reflects internal organizational communication related to management, planning, team coordination, employee support, and risk-related processes.

2. **Trading Data, Products, and Deal Reporting**  
   This topic captures operational trading communication, including product setup, market data, daily reports, desks, and deal-related tracking.

3. **Residual Personal and Low-Information Communication**  
   This topic contains heterogeneous personal or weakly informative email traffic that remained in the corpus even after repeated cleaning. Its presence reflects the mixed nature of the Enron email archive.

4. **California Energy Market and Regulation**  
   This topic captures communication related to the California energy crisis, electricity markets, public regulation, FERC, and state-level intervention.

5. **Gas Contracts, Capacity, and Market Operations**  
   This topic reflects gas-related contracts, pricing, capacity arrangements, customer operations, and pipeline-related market activity.

6. **Legal Agreements and Counterparty Documentation**  
   This topic covers legal and contractual communication, including ISDA agreements, credit-related documentation, and counterparty arrangements.

## 17. Limitations and conclusion

One important limitation of this analysis is that the Enron email corpus combines formal business correspondence with informal, personal, and administrative communication.  
Because of this, even a multi-stage cleaning pipeline cannot fully eliminate all residual noise.

This limitation is visible in the final model, where one topic still reflects mixed low-information or personal communication rather than a fully substantive business theme.  
However, this does not invalidate the analysis. Instead, it reflects the real structure of the dataset and the practical difficulty of applying topic modelling to raw corporate email archives.

Overall, the final model produced a stable and interpretable topic structure.  
The strongest themes were related to internal management, trading operations, energy regulation, gas market activity, and legal agreements.  
This suggests that the Enron dataset is suitable for NLP-based thematic analysis, while also demonstrating the importance of iterative cleaning, model comparison, and qualitative topic validation.